Exploratory notebook: final presentation figures are generated by `python3 run.py` and recorded in `analysis/current/results.json`. Some narrative counts below predate the final date-filter, pricing and cost changes. Use the current report for claims; run the raw workflow and download with `--all` before using reviews/calendars.

# Data understanding

**Rubric: Data Understanding (5 pts).** Business problem, dataset and source, relevant variables, data-quality checks, limitations.

- What's in the snapshot, what we removed and why (`filters`)
- Missing values, odd values, the 22 vs 23 neighbourhood count
- Occupancy and revenue = Inside Airbnb's estimates for the 12 months before 15 June 2026

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
T = config.TABLES   # run `python run.py` from the repo root first if outputs/ is empty

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
%matplotlib inline

## Data
- `raw`: listings before cleaning
- `df`: cleaned listings (one row per listing, with our revenue columns)
- `filters`: rows removed by each cleaning step

In [ ]:
raw = pd.read_csv(config.RAW / "listings.csv.gz", low_memory=False)  # before cleaning
df = pd.read_csv(config.PROCESSED / "listings_clean.csv", low_memory=False)  # cleaned listings
filters = pd.read_csv(T / "filters_applied.csv")

### Columns we use
| Column | What it means | In our clean data |
|---|---|---|
| `neighbourhood_cleansed` | Which neighbourhood | 23 neighbourhoods |
| `room_type` | Whole place or a room | 3,757 whole places, 677 private rooms |
| `bucket` | Our simple type (we made this) | house 2,033 · condo 1,656 · suite 705 |
| `bedrooms` | Number of bedrooms | Typical is 2 |
| `price_num` | Nightly price (we turned the text into a number) | Typical is $379 |
| `number_of_reviews_ltm` | Reviews in the last 12 months (how we measure demand) | Typical is 12 |
| `is_active` | Reviewed in the last year (we made this) | 85% are active |
| `occupancy` | Share of the year booked (Inside Airbnb estimate ÷ 365) | Typical active listing: 102 nights (28%) |
| `revenue` | Yearly revenue (Inside Airbnb estimate) | Typical is about $37k a year |

In [ ]:
cols = ["id", "neighbourhood_cleansed", "room_type", "bucket", "bedrooms", "price_num",
        "number_of_reviews_ltm", "is_active", "occupancy", "revenue"]
small = df[cols]
small.head()

### Data-quality problems (selected columns)
| Problem | How big | What we do |
|---|---|---|
| `bedrooms` is missing | 386 listings (9%) | Assume 1 bedroom (likely studios) |
| No reviews in the last year | 15% of listings | Revenue = $0, marked not active |
| Price is the World Cup week quote | All listings | Can't fix; name it as a risk |
| Occupancy and revenue are estimates, not real | All listings | Inside Airbnb's estimate (from reviews); we say so on the slide |

Data period: occupancy and revenue cover the 12 months before 15 June 2026. Other columns have no missing values. Prices: $64 to $1,974 a night, typical $379.

## Quick look

In [ ]:
filters

In [ ]:
raw.isna().mean().sort_values(ascending=False).head(15)

## Analysis
_Start here._

In [ ]:
small.head(15)